In [3]:
from pathlib import Path
import pandas as pd
import re
from datetime import datetime

# ==========================
# 기본 경로 설정 (필요시 여기만 수정)
# ==========================
BASE_DIR = Path(r"C:\Users\user\Desktop\RAW_LOG")   # 상위 경로
VISION_FOLDER_NAME = "Vision3"                      # 중간 폴더
HISTORY_CSV_PATH = Path(
    r"C:\Users\user\Desktop\a3_vision_json_table_processing_history.csv"
)  # 이력 파일 (추후 경로만 바꿔 쓰면 됨)


def load_history(history_path: Path):
    """
    이력 CSV 읽어서 이미 처리한 파일 목록(set)과 DataFrame 반환.
    파일이 없으면 빈 set / 빈 DataFrame 반환.
    """
    if not history_path.exists():
        return set(), pd.DataFrame(
            columns=["file_path", "vision", "barcode_information", "processed_at"]
        )

    df = pd.read_csv(history_path, dtype=str)
    processed_files = set(df["file_path"].tolist())
    return processed_files, df


def parse_barcode_line(line: str) -> str:
    """
    5번째 줄 예시:
    'Barcode information     : BA1WJ25273503624SJ8T-14F014-AE'
    → 'BA1WJ25273503624SJ8T-14F014-AE'
    """
    m = re.search(r"Barcode information\s*:\s*(.*)", line)
    if m:
        return m.group(1).strip()
    return ""


def parse_program_line(line: str) -> str:
    """
    6번째 줄 예시:
    'Test Program            : LED1' → Vision1
    'Test Program            : LED2' → Vision2
    기타 → 그대로 리턴
    """
    m = re.search(r"Test Program\s*:\s*(.*)", line)
    if not m:
        return ""
    prog = m.group(1).strip()
    if prog == "LED1":
        return "Vision1"
    elif prog == "LED2":
        return "Vision2"
    else:
        return prog  # 혹시 다른 프로그램명이면 그대로 저장


def parse_data_lines(lines):
    """
    19번째 줄부터 마지막까지의 데이터 라인 파싱
    예시:
    '1.01 Test Input Voltage(V)              ,           14.67,           14.60,           14.80,          [PASS]'
    → index: '1.01 Test Input Voltage(V)'
       value: '14.67'
       min: '14.60'
       max: '14.80'
       result: '[PASS]'

    - 첫 콤마 이전 부분을 step description으로 사용
    - step description 내부의 2개 이상 연속 공백은 1개로 축소
    - value/min/max/result는 모두 문자열
    """
    index_list = []
    rows = []

    for raw_line in lines:
        line = raw_line.strip("\n\r")
        if not line.strip():
            continue
        if "," not in line:
            continue

        parts = [p.strip() for p in line.split(",")]

        if len(parts) < 2:
            continue

        # 1) step description (첫 번째 콤마 이전)
        raw_desc = parts[0]
        # 2개 이상 공백 → 1개로 축소
        desc = re.sub(r"\s{2,}", " ", raw_desc).strip()

        # 2) value / min / max / result (모두 문자열)
        value = parts[1] if len(parts) > 1 else ""
        min_val = parts[2] if len(parts) > 2 else ""
        max_val = parts[3] if len(parts) > 3 else ""
        result = parts[4] if len(parts) > 4 else ""

        index_list.append(str(desc))
        rows.append(
            {
                "value": str(value),
                "min": str(min_val),
                "max": str(max_val),
                "result": str(result),
            }
        )

    if not rows:
        return pd.DataFrame(columns=["value", "min", "max", "result"])

    df = pd.DataFrame(rows, index=index_list)
    return df


def process_one_file(file_path: Path):
    """
    개별 Vision 로그 파일 파싱.
    - 5번째 줄: Barcode information
    - 6번째 줄: Test Program → Vision1/2 결정
    - 19번째 줄 이후: test step 데이터
    JSON 파일은 생성하지 않고, 메타정보 + step DataFrame만 반환.
    """
    with file_path.open("r", encoding="cp949", errors="ignore") as f:
        lines = f.readlines()

    if len(lines) < 19:
        # 디버깅용으로만 간단히 표시 (경로는 안 찍어도 됨)
        print("[SKIP] 라인 수 부족")
        return None, None

    barcode_line = lines[4]
    program_line = lines[5]
    data_lines = lines[18:]  # 19번째 줄부터 끝까지

    barcode = parse_barcode_line(barcode_line)
    vision = parse_program_line(program_line)
    df_steps = parse_data_lines(data_lines)

    info = {
        "file_path": str(file_path),  # 이건 이력용으로만 사용
        "vision": vision,
        "barcode_information": barcode,
    }
    return info, df_steps


def run_vision_processing():
    """
    Vision3 로그 전체를 돌면서:
      - 이력에 없는 파일만 파싱
      - 파일별 step 정보를 모아서 하나의 pandas DataFrame으로 반환
      - 이력 CSV(a3_vision_json_table_processing_history.csv)는 누적으로 업데이트

    반환 DataFrame 컬럼:
      [vision, barcode_information, step_description, value, min, max, result]
    """
    vision_root = BASE_DIR / VISION_FOLDER_NAME
    if not vision_root.exists():
        print(f"[ERROR] Vision 폴더가 없습니다: {vision_root}")
        return pd.DataFrame()

    # 이력 로드
    processed_set, history_df = load_history(HISTORY_CSV_PATH)

    target_files = []

    # yyyymmdd 폴더 순회
    for date_dir in sorted(vision_root.iterdir()):
        if not date_dir.is_dir():
            continue

        for sub_name in ["GoodFile", "BadFile"]:
            sub_dir = date_dir / sub_name
            if not sub_dir.exists():
                continue

            for file_path in sorted(sub_dir.glob("*")):
                if file_path.is_file():
                    target_files.append(file_path)

    total = len(target_files)
    print(f"총 대상 파일 수: {total}개")

    new_history_rows = []
    combined_rows = []

    processed_count = 0
    skipped_count = 0

    for idx, file_path in enumerate(target_files, start=1):
        file_str = str(file_path)

        # 이미 이력에 있는 파일이면 PASS
        if file_str in processed_set:
            skipped_count += 1
        else:
            info, df_steps = process_one_file(file_path)

            if info is None or df_steps is None or df_steps.empty:
                skipped_count += 1
            else:
                processed_count += 1

                # 이력용 정보
                history_row = {
                    "file_path": info["file_path"],
                    "vision": info["vision"],
                    "barcode_information": info["barcode_information"],
                    "processed_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                }
                new_history_rows.append(history_row)

                # 최종 테이블용 row들 (file_path는 테이블에서 제외)
                for step_desc, row in df_steps.iterrows():
                    combined_rows.append(
                        {
                            "vision": info["vision"],
                            "barcode_information": info["barcode_information"],
                            "step_description": step_desc,
                            "value": row.get("value", ""),
                            "min": row.get("min", ""),
                            "max": row.get("max", ""),
                            "result": row.get("result", ""),
                        }
                    )

        # 🔹 진행 현황: 1000개 단위 + 마지막에만 출력, 경로 X
        if idx % 1000 == 0 or idx == total:
            print(f"[진행 현황] {idx}/{total} 파일 처리 (신규:{processed_count}, 스킵:{skipped_count})")

    # 이력 갱신 (누적)
    if new_history_rows:
        new_df = pd.DataFrame(new_history_rows)
        updated_history_df = pd.concat([history_df, new_df], ignore_index=True)
        updated_history_df.to_csv(HISTORY_CSV_PATH, index=False, encoding="utf-8-sig")
        print(f"\n이력 파일 업데이트 완료: {HISTORY_CSV_PATH}")
        print(f"  → 새로 처리된 파일 수: {processed_count}개")
    else:
        print("\n새로 처리된 파일이 없습니다. (모두 이력에 존재 또는 파싱 실패)")

    # 최종 결과 DataFrame 생성 후 반환
    if not combined_rows:
        print("결과 데이터가 없습니다.")
        return pd.DataFrame(
            columns=[
                "vision",
                "barcode_information",
                "step_description",
                "value",
                "min",
                "max",
                "result",
            ]
        )

    result_df = pd.DataFrame(combined_rows)[
        ["vision", "barcode_information", "step_description", "value", "min", "max", "result"]
    ]
    return result_df


In [4]:
result_df = run_vision_processing()

# 🔸 결과 테이블 30개까지만 표시
result_df.head(30)

총 대상 파일 수: 17297개
[진행 현황] 1000/17297 파일 처리 (신규:1000, 스킵:0)
[진행 현황] 2000/17297 파일 처리 (신규:2000, 스킵:0)
[진행 현황] 3000/17297 파일 처리 (신규:3000, 스킵:0)
[진행 현황] 4000/17297 파일 처리 (신규:4000, 스킵:0)
[진행 현황] 5000/17297 파일 처리 (신규:5000, 스킵:0)
[진행 현황] 6000/17297 파일 처리 (신규:6000, 스킵:0)
[진행 현황] 7000/17297 파일 처리 (신규:7000, 스킵:0)
[진행 현황] 8000/17297 파일 처리 (신규:8000, 스킵:0)
[진행 현황] 9000/17297 파일 처리 (신규:9000, 스킵:0)
[진행 현황] 10000/17297 파일 처리 (신규:10000, 스킵:0)
[진행 현황] 11000/17297 파일 처리 (신규:11000, 스킵:0)
[진행 현황] 12000/17297 파일 처리 (신규:12000, 스킵:0)
[진행 현황] 13000/17297 파일 처리 (신규:13000, 스킵:0)
[진행 현황] 14000/17297 파일 처리 (신규:14000, 스킵:0)
[진행 현황] 15000/17297 파일 처리 (신규:15000, 스킵:0)
[진행 현황] 16000/17297 파일 처리 (신규:16000, 스킵:0)
[진행 현황] 17000/17297 파일 처리 (신규:17000, 스킵:0)
[진행 현황] 17297/17297 파일 처리 (신규:17297, 스킵:0)

이력 파일 업데이트 완료: C:\Users\user\Desktop\a3_vision_json_table_processing_history.csv
  → 새로 처리된 파일 수: 17297개


,vision,barcode_information,step_description,value,min,max,result
0,Vision2,BA1WJ25273503624SJ8T-14F014-AE,x1,0.00,0.00,0.00,[PASS]
1,Vision2,BA1WJ25273503624SJ8T-14F014-AE,y1,0.00,0.00,0.00,[PASS]
2,Vision2,BA1WJ25273503624SJ8T-14F014-AE,intensity1,0.00,0,0.1,[PASS]
3,Vision2,BA1WJ25273503624SJ8T-14F014-AE,x2,0.00,0.00,0.00,[PASS]
4,Vision2,BA1WJ25273503624SJ8T-14F014-AE,y2,0.00,0.00,0.00,[PASS]
5,Vision2,BA1WJ25273503624SJ8T-14F014-AE,intensity2,0.00,0,0.1,[PASS]
6,Vision2,BA1WJ25273503624SJ8T-14F014-AE,x3,0.00,0.00,0.00,[PASS]
7,Vision2,BA1WJ25273503624SJ8T-14F014-AE,y3,0.00,0.00,0.00,[PASS]
8,Vision2,BA1WJ25273503624SJ8T-14F014-AE,intensity3,0.00,0,0.1,[PASS]
9,Vision2,BA1WJ25273503624SJ8T-14F014-AE,x4,0.00,0.00,0.00,[PASS]
